In [3]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp

# 1. Schemat (upewnij się, że jest w tej samej komórce lub był uruchomiony wcześniej)
json_schema = StructType([
    StructField("Lines", StringType(), True),
    StructField("VehicleNumber", StringType(), True),
    StructField("Brigade", StringType(), True),
    StructField("Lat", StringType(), True),
    StructField("Lon", StringType(), True),
    StructField("Time", StringType(), True)
])

# 2. Ścieżki
gps_path = "Files/input_data/gps/"
checkpoint_path = "Files/_checkpoints/gps_ingestion/"

# 3. Odczyt strumieniowy
df_gps_stream = (spark.readStream
    .format("json")
    .schema(json_schema)
    .option("multiline", "true")
    .load(gps_path))

# 4. ZAPIS - tutaj była zmiana z .table() na .toTable()
query = (df_gps_stream.withColumn("ingestion_time", current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append") # Określamy, że dopisujemy nowe dane
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("bronze_gps")) # Zmienione na .toTable()

query.awaitTermination()
print("Sukces! Warstwa BRONZE została zasilona danymi z plików.")


StatementMeta(, 8aa45a17-c416-4c9c-b351-22228dda61e8, 5, Finished, Available, Finished, False)

Sukces! Warstwa BRONZE została zasilona danymi z plików.


In [4]:
# Odczytuje tabelę i wyświetla ją w formie ładnej, klikalnej siatki
display(spark.read.table("bronze_gps"))

StatementMeta(, 8aa45a17-c416-4c9c-b351-22228dda61e8, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 31d38bc0-1099-46ff-bdda-8d245f365cc3)